In [2]:
import numpy as np
import pandas as pd

In [8]:
spot_df = pd.read_csv('../../data/Spotify_Music.csv')
spot_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   artists           113999 non-null  object 
 1   album_name        113999 non-null  object 
 2   track_name        113999 non-null  object 
 3   popularity        114000 non-null  int64  
 4   duration_ms       114000 non-null  int64  
 5   explicit          114000 non-null  bool   
 6   danceability      114000 non-null  float64
 7   energy            114000 non-null  float64
 8   key               114000 non-null  int64  
 9   loudness          114000 non-null  float64
 10  mode              114000 non-null  int64  
 11  speechiness       114000 non-null  float64
 12  acousticness      114000 non-null  float64
 13  instrumentalness  114000 non-null  float64
 14  liveness          114000 non-null  float64
 15  valence           114000 non-null  float64
 16  tempo             11

In [9]:
# target column is popularity, bucket since 0-100
spot_df['Popularity'] = pd.cut(
    spot_df['popularity'],
    bins=[0, 25, 50, 75, 100],
    labels=['Fresh', 'New On Repeat', 'Can\'t Get Enough', 'Top Hit of Today'],
    right=False,
    include_lowest=True
)
spot_df[['popularity', 'Popularity']].head(10)

,popularity,Popularity
0,73,Can't Get Enough
1,55,Can't Get Enough
2,57,Can't Get Enough
3,71,Can't Get Enough
4,82,Top Hit of Today
5,58,Can't Get Enough
6,74,Can't Get Enough
7,80,Top Hit of Today
8,74,Can't Get Enough
9,56,Can't Get Enough


In [10]:
spot_df.to_csv(f'../../data/Spotify_Music_Cleaned.csv', index=False)

In [7]:
import lightgbm as lgb 

#check if the lgb is working 
print(lgb.__version__)

4.6.0


In [1]:
import os
import psutil
import joblib

print(f"CPU cores (logical):    {os.cpu_count()}")
print(f"CPU cores (physical):   {psutil.cpu_count(logical=False)}")
print(f"joblib default backend: {joblib.effective_n_jobs(-1)} workers")

# Check what's actually being used during a cross_val_score call
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
import numpy as np

X = np.random.rand(1000, 10)
y = np.random.randint(0, 2, 1000)

import time
t0 = time.perf_counter()
cross_val_score(GradientBoostingClassifier(n_estimators=50), X, y, cv=5, n_jobs=1)
t1 = time.perf_counter()
cross_val_score(GradientBoostingClassifier(n_estimators=50), X, y, cv=5, n_jobs=-1)
t2 = time.perf_counter()

print(f"\ncross_val_score n_jobs=1:  {t1-t0:.2f}s")
print(f"cross_val_score n_jobs=-1: {t2-t1:.2f}s")

CPU cores (logical):    10
CPU cores (physical):   10
joblib default backend: 10 workers

cross_val_score n_jobs=1:  0.34s
cross_val_score n_jobs=-1: 0.86s


In [2]:
import optuna
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import time

X = np.random.rand(5000, 10)
y = np.random.randint(0, 2, 5000)

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 2, 7)
    clf = GradientBoostingClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    return float(np.mean(cross_val_score(clf, X, y, cv=5, n_jobs=1)))

# Sequential trials
t0 = time.perf_counter()
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10, show_progress_bar=False, n_jobs=1)
t1 = time.perf_counter()
print(f"Optuna n_jobs=1  (sequential trials): {t1-t0:.2f}s")

# Parallel trials
t0 = time.perf_counter()
study2 = optuna.create_study(direction="maximize")
study2.optimize(objective, n_trials=10, show_progress_bar=False, n_jobs=5)
t2 = time.perf_counter()
print(f"Optuna n_jobs=5  (parallel trials):   {t2-t0:.2f}s")

/Users/dg/Documents/Project_Local_Class_Engine/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-04-23 23:53:33,966] A new study created in memory with name: no-name-c693f602-3445-42c7-8e43-d2f370f684a5
[I 2026-04-23 23:53:37,370] Trial 0 finished with value: 0.5147999999999999 and parameters: {'n_estimators': 60, 'max_depth': 5}. Best is trial 0 with value: 0.5147999999999999.
[I 2026-04-23 23:53:40,900] Trial 1 finished with value: 0.5076 and parameters: {'n_estimators': 102, 'max_depth': 3}. Best is trial 0 with value: 0.5147999999999999.
[I 2026-04-23 23:53:43,737] Trial 2 finished with value: 0.5069999999999999 and parameters: {'n_estimators': 82, 'max_depth': 3}. Best is trial 0 with value: 0.5147999999999999.
[I 2026-04-23 23:53:50,909] Trial 3 finished with value: 0.5122000000000001 a

Optuna n_jobs=1  (sequential trials): 63.88s


[I 2026-04-23 23:54:43,617] Trial 3 finished with value: 0.508 and parameters: {'n_estimators': 86, 'max_depth': 4}. Best is trial 3 with value: 0.508.
[I 2026-04-23 23:54:43,883] Trial 1 finished with value: 0.5048 and parameters: {'n_estimators': 119, 'max_depth': 3}. Best is trial 3 with value: 0.508.
[I 2026-04-23 23:54:43,905] Trial 0 finished with value: 0.5045999999999999 and parameters: {'n_estimators': 118, 'max_depth': 3}. Best is trial 3 with value: 0.508.
[I 2026-04-23 23:54:48,270] Trial 2 finished with value: 0.5122000000000001 and parameters: {'n_estimators': 156, 'max_depth': 4}. Best is trial 2 with value: 0.5122000000000001.
[I 2026-04-23 23:54:51,085] Trial 7 finished with value: 0.507 and parameters: {'n_estimators': 135, 'max_depth': 3}. Best is trial 2 with value: 0.5122000000000001.
[I 2026-04-23 23:54:53,668] Trial 5 finished with value: 0.5132 and parameters: {'n_estimators': 83, 'max_depth': 7}. Best is trial 5 with value: 0.5132.
[I 2026-04-23 23:54:55,056] T

Optuna n_jobs=5  (parallel trials):   24.87s
